# Pillar 3I-a — Vetted-Override / Decision-Distillation on V15 (pillar3f)

**A different lever than 3i.** 3i keeps the *soft* 3b recipe and tunes only the learning rate.
This notebook changes the **target itself**: instead of distilling the soft visit/Q distribution
over all states, it does **hard BC toward the search's actual decision (argmax visits)** —
but **only on the states where that decision overrides the prior's argmax**.

**The logic (reviewer's spec):** treat the 400-sim search as a *vetting* process.
- `argmax(visits) == argmax(prior)` -> the prior was already good enough at this budget -> **skip** the state (weight 0, or a tiny prior-anchor).
- `argmax(visits) != argmax(prior)` -> the search overrode the prior -> a **high-SNR prior-blunder** signal -> **hard one-hot BC** toward the search's move.

**Why this might beat soft distillation:** soft visit/Q distillation *smears* — on the ~75% of states the prior already gets right it teaches slightly *less* confidence (re-sharpening risk, the exact failure mode that regressed 3g). Decision-distillation touches only the ~25% the search actually changed its mind on, and teaches a clean, confident correction there.

This is **distinct from completed-Q (3g2)**, which distilled a *rejected low-visit* move. Here the target is the teacher's **played** move = `argmax(visits)`.

**Override rate on V15 ~= 24.8%** (measured on the slim tensor). Watch `corr_match` (fraction of override states where the student now picks the override move) rise across epochs — that's the BC taking hold. **But eval by GAMEPLAY floor, not `corr_match`** (high corr_match just means BC memorized; it does not guarantee a better floor).

**Eval bar to beat** (pillar3f, 1000 seeds 779000-779999, uncapped): **mean 33837, P1 968, P5 2457, P10 4507, P25 11142, P50 24552, <1000 1.1%**.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_pillar3d_v2.tar.gz` — current code (has `train_gumbel.py` + `gumbel.py` + `dataset.py` with `vetted_override`).
2. `v15_pillar3f_slim.pt.gz` — the V15 slim tensor (gzip; carries `cand_visit` + `cand_prior`, the override inputs).
3. `pillar3f.pt` — warm-start weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3f.pt', '/content/alphatrain/data/pillar3f.pt')
print(f'Warm-start (pillar3f): {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')

# Copy gzipped tensor to local disk FIRST, verify integrity, THEN decompress.
# (Streaming through Drive FUSE can silently truncate files this large.)
t0 = time.time()
!cp {DRIVE}/v15_pillar3f_slim.pt.gz /content/v15_pillar3f_slim.pt.gz
print(f'.gz copied: {os.path.getsize("/content/v15_pillar3f_slim.pt.gz"):,} bytes')
!gunzip -t /content/v15_pillar3f_slim.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/v15_pillar3f_slim.pt.gz > /content/alphatrain/data/v15_pillar3f_slim.pt
pt_size = os.path.getsize('/content/alphatrain/data/v15_pillar3f_slim.pt')
EXPECTED_PT = 2_047_271_567
assert pt_size == EXPECTED_PT, (
    f'Decompressed .pt size wrong! got {pt_size} expected {EXPECTED_PT}. '
    f'Re-upload v15_pillar3f_slim.pt.gz to Drive.')
print(f'V15 slim tensor: {pt_size/1e9:.2f} GB ({time.time()-t0:.0f}s)')
!rm /content/v15_pillar3f_slim.pt.gz

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG =====
# Primary run = the cleanest reading of the reviewer's spec:
#   --anchor-mode partition  -> hard BC ONLY on override states, prior-anchor ONLY on agreement states
#   --beta 0.1               -> the "tiny KL-anchor to prior" on the agreement (kept) states
LR          = 1e-4            # gentle (strong-policy regime); bump/drop if floor doesn't move
BETA        = 0.1            # prior anchor strength on the kept (agreement) states
ANCHOR_MODE = 'partition'    # 'partition' = BC-on-override + anchor-on-agreement (reviewer-faithful)
MIN_MARGIN  = 0.0            # 0 = pure argmax disagreement; >0 keeps only confident overrides
RUN         = 'pillar3i_a_part_b0.1'
EPOCHS      = 12             # hard BC + warm-start converges fast; save & eval every epoch
print(f'RUN={RUN}  LR={LR}  BETA={BETA}  ANCHOR={ANCHOR_MODE}  MIN_MARGIN={MIN_MARGIN}  EPOCHS={EPOCHS}')

# --- Variants to try if the primary run doesn't lift the floor (change above + rename RUN) ---
#  Drop agreement entirely (no anchor):      ANCHOR_MODE='everywhere', BETA=0.0
#  Tiny anchor on ALL states (incl override):ANCHOR_MODE='everywhere', BETA=0.1
#  Confident overrides only:                 MIN_MARGIN=0.10  (suppresses flat coin-flip overrides)
# NOTE: in 'partition' mode BETA=0 is treated as anchor-weight 1.0 (NOT off);
#       use 'everywhere'+BETA=0 for the true drop-agreement variant.

In [ ]:
# Pillar 3I-a = vetted-override decision distillation on V15, warm-started from pillar3f.
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=. python -m alphatrain.train_gumbel \
    --tensor-file alphatrain/data/v15_pillar3f_slim.pt \
    --resume alphatrain/data/pillar3f.pt --warm-start \
    --target-mode vetted_override --min-margin {MIN_MARGIN} \
    --anchor-mode {ANCHOR_MODE} --beta {BETA} --ce-mode candidate \
    --epochs {EPOCHS} --batch-size 16384 --lr {LR} --warmup-epochs 1 \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN}
# WATCH corr%~25 (override rate) and corr_match rising (BC taking). val loss != gameplay.

In [ ]:
# Copy every epoch checkpoint to Drive (val/corr_match are weak predictors -> eval gameplay per-epoch).
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst); print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}'); print(f'Saved {DRIVE}/{RUN}_{f}')

## Eval (per epoch — gameplay floor, NOT val/corr_match)
Pull each epoch checkpoint back and run `eval_policy` on the fixed seed range, distribution-level:
```
python -m scripts.eval_policy --model <ckpt> --device cuda --batch 1024 \
    --seed-start 779000 --seed-end 779999
```
Compare the **floor (P1, P5, P10, P25, <1000)** against pillar3f: **mean 33837, P1 968, P5 2457, P10 4507, P25 11142, P50 24552, <1000 1.1%**.
- Any epoch with floor **>= pillar3f** -> decision-distillation recovered improvement; pick the best, log to HISTORY.
- All epochs **< pillar3f** (like 3g/3g2) -> hard-BC-on-overrides also degrades -> the override set itself is not a clean improvement signal at 400 sims (back to the reviewers; the deeper-MCTS teacher relabel is the fallback).
- `corr_match` -> ~1.0 but floor **drops** -> BC memorized the overrides but they were not real improvements (over-trusting the 400-sim decision). Try `MIN_MARGIN=0.10` (confident overrides only).